# Faruq-v3 AF2R — Kaggle one-click screening

Tambahkan satu **private Kaggle Dataset** yang berisi tiga file berikut:

1. `faruq-development-v3-grouped.tar`
2. checkpoint AF2 `best.pt` (boleh dinamai `AF2_seed42_best.pt`)
3. `lfdet_afab_seed42_screening.json`

Aktifkan GPU dan Internet. Untuk eksekusi tanpa menjaga browser, pilih **Save Version → Save & Run All**. Notebook menjalankan static audit, AF2R0, AF2R1, lalu decision. Test tidak tersedia dan tidak dibaca.


In [ ]:
import importlib,json,os,shutil,subprocess,sys,tarfile,time,torch
from pathlib import Path
assert torch.cuda.is_available(),'Aktifkan GPU pada Notebook options.'
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
def unique(name):
    matches=sorted(path for path in INPUT.rglob(name) if path.is_file())
    if len(matches)!=1: raise FileNotFoundError(f'Harus ada tepat satu {name}; ditemukan {matches}')
    return matches[0]
ARCHIVE=unique('faruq-development-v3-grouped.tar')
af2_named=sorted(path for path in INPUT.rglob('AF2_seed42_best.pt') if path.is_file())
AF2=af2_named[0] if len(af2_named)==1 else unique('best.pt')
REFERENCE=unique('lfdet_afab_seed42_screening.json')
REPO=WORK/'coffee-bean-detection'; BRANCH='agent/af2-adaptive-residual-gate'
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
DATA=WORK/'faruq-development-v3-grouped'
if not (DATA/'data.yaml').is_file():
    with tarfile.open(ARCHIVE,'r') as archive: archive.extractall(WORK,filter='data')
assert (DATA/'data.yaml').is_file() and not (DATA/'test').exists()
OUTPUT=WORK/'faruq-v3-af2-adaptive-residual-v1'
if not OUTPUT.exists():
    previous=[path for path in INPUT.rglob(OUTPUT.name) if path.is_dir() and (path/'static_audit.json').is_file()]
    if len(previous)==1:
        print('RESTORE OUTPUT SEBELUMNYA:',previous[0]); shutil.copytree(previous[0],OUTPUT)
print('GPU:',torch.cuda.get_device_name(0)); print('ARCHIVE:',ARCHIVE); print('AF2:',AF2); print('OUTPUT:',OUTPUT)


In [ ]:
from coffee_detector.af2r.audit import run_af2r_static_audit
STATIC=OUTPUT/'static_audit.json'
audit=run_af2r_static_audit(AF2,STATIC,device='cuda:0')
print('STATIC PARAMETERS:',{'source':audit['source_parameters'],'candidate':audit['candidate_parameters'],'added':audit['added_parameters']})
print('STATIC GATES:',audit['gates']); print('STATIC DECISION:',audit['decision'])
assert audit['decision']=='PASS','STOP: static audit gagal; training tidak dijalankan.'


In [ ]:
def run_arm(arm):
    result_path=OUTPUT/'val_reports'/f'{arm}_seed42_result.json'
    if result_path.is_file():
        print('REUSE SELESAI:',arm); return json.loads(result_path.read_text())
    command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2r_arm','--arm',arm,'--data-root',str(DATA),'--grouped-summary',str(DATA/'faruq_grouped_summary.json'),'--af2-checkpoint',str(AF2),'--static-audit',str(STATIC),'--output-root',str(OUTPUT),'--seed','42','--device','0','--authorize-training']
    log=OUTPUT/f'{arm}_seed42_run.log'; log.parent.mkdir(parents=True,exist_ok=True)
    print('START:',arm,'| log=',log,flush=True)
    with log.open('a',encoding='utf-8') as handle:
        process=subprocess.Popen(command,cwd=REPO,stdout=handle,stderr=subprocess.STDOUT)
        while process.poll() is None:
            try: process.wait(timeout=300)
            except subprocess.TimeoutExpired:
                csv=OUTPUT/arm/f'{arm}_seed42/results.csv'; epochs=max(0,len(csv.read_text(errors='replace').splitlines())-1) if csv.is_file() else 0
                print(f'{arm}: {epochs}/30 epoch tercatat',flush=True)
    if process.returncode:
        print('\n'.join(log.read_text(errors='replace').splitlines()[-150:])); raise RuntimeError(f'{arm} gagal: {process.returncode}')
    print('SELESAI:',arm); return json.loads(result_path.read_text())
results={arm:run_arm(arm) for arm in ('AF2R0','AF2R1')}
print({arm:{k:v for k,v in result['metrics'].items() if k in ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')} for arm,result in results.items()})


In [ ]:
from coffee_detector.experiments.run_faruq_v3_af2r_decision import run_faruq_v3_af2r_decision
decision=run_faruq_v3_af2r_decision(OUTPUT,REFERENCE,seed=42)
import pandas as pd
from IPython.display import display
rows=[{'model':model,**metrics} for model,metrics in decision['values'].items()]
display(pd.DataFrame(rows).style.format({c:'{:.2%}' for c in ['macro_map50_95','bottom3_class_map50_95','worst_class_map50_95']}))
print('AF2R1 vs AF2R0:',decision['af2r1_minus_af2r0']); print('AF2R1 vs AF2:',decision['af2r1_minus_fixed_af2']); print('CRITERIA:',decision['criteria']); print('DECISION:',decision['decision']); print('NEXT:',decision['next']); print('TEST:',decision['test_opened'])


In [ ]:
archive_path=shutil.make_archive(str(WORK/'faruq-v3-af2r-screening-output'),'zip',root_dir=OUTPUT)
print('OUTPUT FOLDER:',OUTPUT); print('DOWNLOAD ZIP:',archive_path)
print('Simpan versi notebook ini. Seluruh checkpoint, log, report, dan decision menjadi Kaggle Output.')
